# 14 — Refusal + Eval-Awareness Directions on Llama 3.3 70B

Self-contained notebook that fits **two** kinds of mean-difference directions on **`meta-llama/Llama-3.3-70B-Instruct`** — the same model used for `exp06_lamma/`. Just the vectors, no JailbreakBench gen, no LlamaGuard eval, no CE-loss sweep.

**(A) Refusal direction** — Arditi et al. *Refusal in Language Models Is Mediated by a Single Direction* (arXiv 2406.11717), recipe simplified to the candidate-generation step:
1. Load 128 harmful + 128 harmless training instructions from `third_party/refusal_direction/dataset/splits/`.
2. Apply Llama 3.3 chat template with `enable_thinking=False` (otherwise Qwen3.5 prepends a `<think>` block that misaligns the EOI positions).
3. Forward pass; capture the post-block residual stream at the **last 5 end-of-instruction token positions** (`pos = -5 … -1`) across **every layer**.
4. Per (pos, layer): `direction = mean(harmful) − mean(harmless)` → unit-normalized.
5. Save the full `(n_pos=5, n_layers, d_model)` candidate cube. We skip Arditi's selection step (it picks one (pos, layer) by ablation effect) — for downstream comparison work we want all candidates.

**(B) Eval-awareness directions** — Nguyen et al. *Probing and Steering Evaluation Awareness of Language Models* (arXiv 2507.01786), 4 contrastive datasets:
1. For each item `{question, test, deploy}` build `f"{question}\n\n {answer}"` for each side.
2. Forward pass; capture post-block residual stream at the position of the last `A`/`B` answer-character token at every layer.
3. Per (dataset, layer): `direction = mean(test) − mean(deploy)` → unit-normalized.

**Hook convention.** Both extractions use `register_forward_hook` on `model.model.layers[L]` — i.e. the **output** of block L, post-residual. This matches `mech_spoof.activations._register_hooks` and `exp06_lamma/` so the new directions are layer-aligned with the existing S/U directions.

**Outputs (to `OUT_DIR`):**
- `directions.npz` — all directions, keys:
  - `refusal__mm_dir__pos_{P}__layer_{LLL}` — refusal mean-diff (unit-norm), `(d_model,)`. `P ∈ {-5..-1}`.
  - `refusal__mm_raw__pos_{P}__layer_{LLL}` — un-normalized harmful − harmless.
  - `refusal__harmful_mean__pos_{P}__layer_{LLL}` / `refusal__harmless_mean__...` — class means.
  - `eval_aw__mm_dir__{dataset}__layer_{LLL}` — eval-aw mean-diff (unit-norm).
  - `eval_aw__mm_raw__{dataset}__layer_{LLL}` / `eval_aw__test_mean__...` / `eval_aw__deploy_mean__...`.
- `manifest.json` — per-direction stats (`mm_natural_scale`, `frac_class_proj_larger`, `n_used`, `n_skipped`).

**Compute.** Llama 3.3 70B in bf16 fits on a single 16 GB GPU. With 128 harmful + 128 harmless + ~1500 eval-aw items × 2 prompts ≈ 3,250 forwards, expect **40–90 min** wall-clock on an H100 / A100 80GB, longer on smaller GPUs.

**Tokens needed.** None — `meta-llama/Llama-3.3-70B-Instruct` is not gated.

## 0 — Sanity: GPU check

In [ ]:
import subprocess, torch
try:
    print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print('no nvidia-smi (CPU/MPS host?)')
if torch.cuda.is_available():
    n = torch.cuda.device_count()
    total_gb = sum(torch.cuda.get_device_properties(i).total_memory for i in range(n)) / 1e9
    print(f'{n} GPUs visible, total VRAM = {total_gb:.0f} GB')
else:
    print('no CUDA; will run on CPU/MPS — extremely slow but correct.')

## 1 — Install dependencies

Most pods/Colab images already have `torch` + `transformers`. Comment out if your env is set up.

In [ ]:
!pip install -q 'transformers>=4.45' 'accelerate>=0.33' huggingface_hub tqdm numpy

## 1b — Clone the Mech_spoof repo (if not already on disk)

On a fresh Colab/pod we need the repo for:
- `src/mech_spoof/` (model loader, template adapter, configs)
- `third_party/refusal_direction/dataset/splits/` (harmful_train.json + harmless_train.json)
- `third_party/eval_awareness_datasets/` (the 4 contrastive JSONs)

All three live inside the main repo (eval-aw datasets are committed alongside the rest), so a single clone covers everything. Skip this cell if you've rsynced the repo already and `MECH_SPOOF_ROOT` is set.

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/ChuloIva/Mech_spoof.git'

# Default clone target: /content/Mech_spoof on Colab, /workspace/Mech_spoof on a pod, cwd otherwise.
if os.environ.get('MECH_SPOOF_ROOT'):
    target = Path(os.environ['MECH_SPOOF_ROOT'])
elif Path('/content').exists():
    target = Path('/content/Mech_spoof')
elif Path('/workspace').exists():
    target = Path('/workspace/Mech_spoof')
else:
    target = Path.cwd() / 'Mech_spoof'

if not (target / 'src' / 'mech_spoof').exists():
    target.parent.mkdir(parents=True, exist_ok=True)
    print(f'cloning {REPO_URL} → {target}')
    !git clone --depth 1 {REPO_URL} {target}
else:
    print(f'repo already at {target} (skipping clone)')

os.environ['MECH_SPOOF_ROOT'] = str(target)

# Quick sanity on the bits we need.
for sub in ['src/mech_spoof',
            'third_party/refusal_direction/dataset/splits',
            'third_party/eval_awareness_datasets']:
    p = target / sub
    print(f'  {sub:<48s} {"OK" if p.exists() else "MISSING"}')

## 2 — Project root, tokens, paths

Works on Colab (mounts Drive, reads `HF_TOKEN` from userdata) or on a pod (just relies on env / repo on disk).

In [ ]:
import os, sys, json
from pathlib import Path

# Resolve project root: env override → /workspace/Mech_spoof → notebook-relative.
PROJECT_ROOT = Path(os.environ.get('MECH_SPOOF_ROOT', '/workspace/Mech_spoof'))
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    print(f'falling back to PROJECT_ROOT={PROJECT_ROOT}')
assert (PROJECT_ROOT / 'src' / 'mech_spoof').exists(), f'no mech_spoof at {PROJECT_ROOT}/src — set MECH_SPOOF_ROOT'

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

REFUSAL_SPLITS_DIR = PROJECT_ROOT / 'third_party' / 'refusal_direction' / 'dataset' / 'splits'
EVAL_AW_DIR        = PROJECT_ROOT / 'third_party' / 'eval_awareness_datasets'
OUT_DIR            = PROJECT_ROOT / 'exp_directions_llama33_70b'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# HF auth — Llama 3.3 70B is gated — paste your `HF_TOKEN` in §1c above.
try:
    from google.colab import userdata
    try:
        os.environ.setdefault('HF_TOKEN', userdata.get('HF_TOKEN'))
    except Exception:
        pass
except Exception:
    pass

print('PROJECT_ROOT       :', PROJECT_ROOT)
print('refusal splits dir :', REFUSAL_SPLITS_DIR, '(exists)' if REFUSAL_SPLITS_DIR.exists() else '(MISSING)')
print('eval-aw datasets   :', EVAL_AW_DIR,        '(exists)' if EVAL_AW_DIR.exists()        else '(MISSING)')
print('out dir            :', OUT_DIR)

## 1c — Paste your Hugging Face token

Llama 3.3 70B is gated. Paste your `HF_TOKEN` between the quotes below and run this cell. The token is stored only in this kernel's environment (not written to disk). If you're on Colab and have already set `HF_TOKEN` in *Secrets*, you can leave this empty.


In [ ]:
import os
from huggingface_hub import login

# >>> Paste your HF token between the quotes <<<
HF_TOKEN_PASTE = ""  # e.g. "hf_AbCdEfGhIjKlMnOpQrStUvWxYz1234567890"

if HF_TOKEN_PASTE:
    os.environ['HF_TOKEN'] = HF_TOKEN_PASTE
if os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN'):
    login(token=os.environ.get('HF_TOKEN') or os.environ['HUGGING_FACE_HUB_TOKEN'],
          add_to_git_credential=False)
    print('logged in to Hugging Face')
else:
    raise RuntimeError('No HF token set. Paste it above (HF_TOKEN_PASTE = "hf_...") or set HF_TOKEN in env / Colab Secrets.')


## 3 — Load Llama 3.3 70B (bf16)

Uses `mech_spoof.models.load_model('llama33_70b')` so the chat-template adapter, layer module path, and dtype match the rest of the codebase. Llama 3.3 70B in bf16 is ~70 GB (8-bit) / ~140 GB (bf16).

In [ ]:
from mech_spoof.models import load_model

# MECH_SPOOF_LLAMA33_MULTI_GPU_LOAD
# Multi-GPU loader. Pick precision below; load_model honours device_map='auto'
# so the 70B is automatically sharded across every visible GPU.
import torch
QUANT = 'bf16'  # default: full bf16 (~140 GB; fits 3x A100 80GB).
                # alternatives: '8bit' (~70 GB, fits 1x 80GB or 3x 40GB), '4bit' (~35 GB).

# Reserve ~4 GB per GPU for activations + KV cache; the rest is for weights.
n_gpus = torch.cuda.device_count()
if n_gpus == 0:
    raise RuntimeError('no CUDA GPUs visible')
per_gpu_gb = [int(torch.cuda.get_device_properties(i).total_memory / 1e9) for i in range(n_gpus)]
HEADROOM_GB = 4
max_memory = {i: f'{max(per_gpu_gb[i] - HEADROOM_GB, 4)}GiB' for i in range(n_gpus)}
max_memory['cpu'] = '32GiB'   # tiny CPU offload safety valve
print(f'visible GPUs: {n_gpus}, per-GPU memory budget: {max_memory}')

# Translate QUANT for load_model: 'bf16' means no quantization (full precision).
_quant_arg = 'none' if QUANT == 'bf16' else QUANT
loaded = load_model(
    'llama33_70b',
    quantization=_quant_arg,
    device_map='auto',
    max_memory=max_memory,
)
model      = loaded.hf_model
tokenizer  = loaded.tokenizer
n_layers   = loaded.n_layers
d_model    = loaded.d_model   # composite configs (e.g. Qwen3.5) don't expose hidden_size at top-level
first_param_device = next(model.parameters()).device
model.eval()
print(f'model loaded: hf_id={loaded.cfg.hf_id} n_layers={n_layers} d_model={d_model} '
      f'device={first_param_device} layers_path={loaded.layers_path}')

# Probe whether tokenizer's chat template accepts enable_thinking (Qwen3+).
supports_thinking = False
try:
    tokenizer.apply_chat_template(
        [{'role': 'user', 'content': 'ping'}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    supports_thinking = True
except (TypeError, ValueError):
    supports_thinking = False
print('chat template supports enable_thinking:', supports_thinking)

# Show how the 70B got sharded across GPUs (sanity check for multi-GPU loads).
from collections import Counter
_dev_counts = Counter()
for name, p in loaded.hf_model.named_parameters():
    _dev_counts[str(p.device)] += 1
print('parameter-tensor counts per device:', dict(_dev_counts))
# First param's device is what `model.generate` / `model(...)` expect inputs on.
print('first param device (input target):', next(loaded.hf_model.parameters()).device)


## 4 — Shared activation extractor

One forward pass; via post-block `register_forward_hook` on `model.model.layers[L]` we capture the residual stream at the requested positions for every layer. Returns `(n_layers, n_positions, d_model)` float32 CPU tensor (~10 MB per call for Llama 3.3 70B with 5 EOI positions).

In [ ]:
import torch

def extract_at_positions(input_ids: torch.Tensor, positions: list[int]) -> torch.Tensor:
    """Forward `input_ids` (1, seq) and return (n_layers, len(positions), d_model) float32 CPU.

    `positions` may contain negative indices (e.g. -1 for last token). We resolve them
    against the actual input length once at hook fire-time per layer.
    """
    seq_len = input_ids.shape[1]
    resolved = [p if p >= 0 else seq_len + p for p in positions]
    assert all(0 <= p < seq_len for p in resolved), f'bad positions {positions} for seq_len={seq_len}'

    storage: list = [None] * n_layers
    handles = []
    for i in range(n_layers):
        layer = loaded.layer_module(i)
        def _hook(_m, _inp, out, idx=i, ps=resolved):
            h = out[0] if isinstance(out, tuple) else out  # (1, seq, d_model)
            storage[idx] = h[0, ps, :].detach().float().cpu()  # (n_positions, d_model)
        handles.append(layer.register_forward_hook(_hook))
    try:
        with torch.no_grad():
            model(input_ids=input_ids.to(first_param_device), use_cache=False)
    finally:
        for h in handles:
            h.remove()
    return torch.stack(storage)  # (n_layers, n_positions, d_model)

## 5 — Refusal direction (Arditi mean-diff)

Apply Llama 3.3 chat template (with `enable_thinking=False`), tokenize, capture residuals at the last 5 EOI positions. Mean over the harmful and harmless training instructions separately, take the difference per (pos, layer), unit-normalize.

We use `cfg.n_train = 128` (Arditi's default). Filtering by refusal score (used in the original pipeline) is skipped — we just want the candidate vectors.

In [ ]:
import random
from tqdm.auto import tqdm

N_TRAIN  = 128         # Arditi default
N_EOI    = 5           # last-N positions of the EOI / generation-prompt tail
POSITIONS = list(range(-N_EOI, 0))   # [-5, -4, -3, -2, -1]

random.seed(42)
harmful_train  = random.sample(
    [d['instruction'] for d in json.loads((REFUSAL_SPLITS_DIR / 'harmful_train.json').read_text())],
    N_TRAIN,
)
harmless_train = random.sample(
    [d['instruction'] for d in json.loads((REFUSAL_SPLITS_DIR / 'harmless_train.json').read_text())],
    N_TRAIN,
)
print(f'sampled {len(harmful_train)} harmful + {len(harmless_train)} harmless train instructions')
print('  harmful  example:', harmful_train[0][:120])
print('  harmless example:', harmless_train[0][:120])

In [ ]:
def render_llama_chat(instruction: str) -> torch.Tensor:
    """Render `instruction` through the Llama 3.3 chat template (no system) with thinking disabled,
    return (1, seq_len) input_ids on CPU.
    """
    extra = {'enable_thinking': False} if supports_thinking else {}
    ids = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': instruction}],
        tokenize=True, add_generation_prompt=True, return_tensors='pt', **extra,
    )
    # Newer transformers may wrap the result in a BatchEncoding (dict-like).
    if hasattr(ids, 'input_ids'):
        ids = ids.input_ids
    elif isinstance(ids, dict) and 'input_ids' in ids:
        ids = ids['input_ids']
    if not torch.is_tensor(ids):
        ids = torch.as_tensor(ids)
    if ids.dim() == 1:
        ids = ids.unsqueeze(0)
    return ids   # (1, seq_len)

# Sanity: peek the templated tail to verify thinking is off and EOI tokens are stable.
_demo = render_llama_chat('What is the capital of France?')
print('templated tail (last 12 tokens):',
      tokenizer.decode(_demo[0, -12:].tolist()))
print('seq_len:', _demo.shape[1])

In [ ]:
def accumulate_means(instructions: list[str], desc: str):
    """Streaming mean accumulator. Returns (n_layers, n_pos, d_model) fp32 mean and per-item activations
    so we can compute the projection-frac sanity stat afterwards."""
    running = torch.zeros(n_layers, len(POSITIONS), d_model, dtype=torch.float64)
    per_item: list[torch.Tensor] = []
    for inst in tqdm(instructions, desc=desc):
        ids = render_llama_chat(inst)
        if ids.shape[1] < N_EOI + 2:
            # Templated prompt shorter than the EOI window we slice — skip rather than wrap.
            continue
        acts = extract_at_positions(ids, POSITIONS)  # (n_layers, n_pos, d_model)
        running += acts.double()
        per_item.append(acts)
    n = len(per_item)
    assert n > 0, f'{desc}: 0 usable items'
    mean = (running / n).float()
    return mean, per_item, n

harmful_mean,  harmful_acts,  n_harmful  = accumulate_means(harmful_train,  'harmful')
harmless_mean, harmless_acts, n_harmless = accumulate_means(harmless_train, 'harmless')
print(f'\nharmful: {n_harmful} usable; harmless: {n_harmless} usable')

In [ ]:
import numpy as np

raw_diff   = harmful_mean - harmless_mean                  # (n_layers, n_pos, d_model)
norms      = raw_diff.norm(dim=-1) + 1e-10                  # (n_layers, n_pos)
refusal_dir = raw_diff / norms[..., None]                   # unit-norm per (layer, pos)

# Sanity stat: per (layer, pos), fraction of items where harmful_proj > harmless_proj.
T = torch.stack(harmful_acts)    # (N_h, n_layers, n_pos, d_model)
D = torch.stack(harmless_acts)
proj_T = (T * refusal_dir[None]).sum(dim=-1)   # (N_h, n_layers, n_pos)
proj_D = (D * refusal_dir[None]).sum(dim=-1)
n_min = min(proj_T.shape[0], proj_D.shape[0])
frac_harmful_proj_larger = (proj_T[:n_min] > proj_D[:n_min]).float().mean(dim=0)   # (n_layers, n_pos)

# Print a few summary points: best (layer, pos) by frac_larger.
flat = frac_harmful_proj_larger.numpy()
best = np.unravel_index(np.argmax(flat), flat.shape)
print(f'best frac_harmful_proj_larger = {flat[best]:.3f} at layer={best[0]} pos={POSITIONS[best[1]]}  '
      f'(scale={norms[best].item():.2f})')
print(f'frac@layer={n_layers//2}, pos=-1: {flat[n_layers//2, -1]:.3f}  '
      f'(scale={norms[n_layers//2, -1].item():.2f})')

## 6 — Eval-awareness directions (Nguyen mean-diff)

Same hook setup, but per-item we look at the position of the last `A` or `B` answer-character token. 4 datasets; per-(dataset, layer) mean-diff direction.

In [ ]:
DATASETS = ['contrastive_dataset', 'AB_ambiguous', 'tom_vs_sharegpt', 'truthful']

def _load(ds_name: str):
    raw = json.loads((EVAL_AW_DIR / f'{ds_name}.json').read_text())
    items = []
    for r in raw:
        if 'positive' in r and 'negative' in r:
            items.append({'question': r['question'], 'test': r['positive'], 'deploy': r['negative']})
        elif 'test' in r and 'deploy' in r:
            items.append({'question': r['question'], 'test': r['test'], 'deploy': r['deploy']})
        else:
            raise KeyError(f'{ds_name}: unexpected keys {list(r.keys())}')
    return items

datasets = {d: _load(d) for d in DATASETS}
for d, items in datasets.items():
    print(f'  {d:<22s} {len(items):>4d} items   sample: test={items[0]["test"]!r}  deploy={items[0]["deploy"]!r}')

TOK_A = tokenizer.convert_tokens_to_ids('A')
TOK_B = tokenizer.convert_tokens_to_ids('B')
assert TOK_A is not None and TOK_B is not None and TOK_A != tokenizer.unk_token_id, \
    f'A/B token-id resolution failed: A={TOK_A} B={TOK_B}'
print(f'TOK_A={TOK_A}  TOK_B={TOK_B}  decode={tokenizer.decode([TOK_A])!r}/{tokenizer.decode([TOK_B])!r}')

In [ ]:
def extract_at_answer_token(prompt_text: str, answer_token_id: int):
    """Forward-pass `prompt_text` raw (no chat template — Nguyen's recipe is on the
    plain `question + answer` string), return (n_layers, d_model) at the LAST occurrence
    of `answer_token_id`. Returns None if absent."""
    enc = tokenizer(prompt_text, return_tensors='pt', add_special_tokens=True)
    input_ids = enc.input_ids
    matches = (input_ids[0] == answer_token_id).nonzero(as_tuple=True)[0]
    if len(matches) == 0:
        return None
    pos = matches[-1].item()
    acts = extract_at_positions(input_ids, [pos])  # (n_layers, 1, d_model)
    return acts.squeeze(1)                          # (n_layers, d_model)

# Smoke test.
_q = datasets['contrastive_dataset'][0]['question']
_ans = datasets['contrastive_dataset'][0]['test']
_tok = TOK_A if 'A' in _ans else TOK_B
_acts = extract_at_answer_token(f'{_q}\n\n {_ans}', _tok)
print(f'smoke: shape={tuple(_acts.shape)}  norm@L0={_acts[0].norm():.2f}  '
      f'norm@Lmid={_acts[n_layers//2].norm():.2f}  norm@Llast={_acts[-1].norm():.2f}')

In [ ]:
# Optional cap for fast smoke run; None = full datasets.
MAX_ITEMS_PER_DATASET = None

eval_aw_results: dict[str, dict] = {}
for ds_name, items in datasets.items():
    if MAX_ITEMS_PER_DATASET:
        items = items[:MAX_ITEMS_PER_DATASET]

    test_sum   = torch.zeros(n_layers, d_model, dtype=torch.float64)
    deploy_sum = torch.zeros(n_layers, d_model, dtype=torch.float64)
    n_used = n_skipped = 0
    test_acts:   list[torch.Tensor] = []
    deploy_acts: list[torch.Tensor] = []

    for it in tqdm(items, desc=ds_name):
        prompt_t = f"{it['question']}\n\n {it['test']}"
        prompt_d = f"{it['question']}\n\n {it['deploy']}"
        tok_t    = TOK_A if 'A' in it['test']   else TOK_B
        tok_d    = TOK_A if 'A' in it['deploy'] else TOK_B
        a_t = extract_at_answer_token(prompt_t, tok_t)
        a_d = extract_at_answer_token(prompt_d, tok_d)
        if a_t is None or a_d is None:
            n_skipped += 1
            continue
        test_sum   += a_t.double()
        deploy_sum += a_d.double()
        n_used += 1
        test_acts.append(a_t)
        deploy_acts.append(a_d)

    assert n_used > 0, f'{ds_name}: 0 usable items'
    test_mean   = (test_sum   / n_used).float()
    deploy_mean = (deploy_sum / n_used).float()
    diff        = test_mean - deploy_mean
    norms_d     = diff.norm(dim=-1) + 1e-10
    direction   = diff / norms_d[:, None]

    T = torch.stack(test_acts)
    D = torch.stack(deploy_acts)
    proj_T = (T * direction[None]).sum(dim=-1)
    proj_D = (D * direction[None]).sum(dim=-1)
    frac_larger = (proj_T > proj_D).float().mean(dim=0).tolist()

    eval_aw_results[ds_name] = {
        'direction':           direction.numpy(),
        'test_mean':           test_mean.numpy(),
        'deploy_mean':         deploy_mean.numpy(),
        'mm_natural_scale':    norms_d.tolist(),
        'frac_test_proj_larger': frac_larger,
        'n_used':              int(n_used),
        'n_skipped':           int(n_skipped),
    }
    mid = n_layers // 2
    print(f'  {ds_name}: n={n_used} skipped={n_skipped}  '
          f'frac_larger@Lmid={frac_larger[mid]:.3f}  scale@Lmid={norms_d[mid]:.3f}')

## 7 — Save NPZ + manifest

One archive holds everything. Keys are namespaced with `refusal__…` and `eval_aw__…` so downstream code can filter by prefix.

In [ ]:
npz_payload: dict = {}
manifest = {
    'experiment':    'refusal_and_eval_awareness_directions',
    'model_key': 'llama33_70b',
    'hf_id':         loaded.cfg.hf_id,
    'n_layers':      n_layers,
    'd_model':       d_model,
    'hook':          'post-block forward hook on model.model.layers[L]',
    'enable_thinking_supported': supports_thinking,
    'refusal': {
        'recipe':            'Arditi et al. 2406.11717 — mean-diff at last 5 EOI positions',
        'positions':         POSITIONS,
        'n_train_per_class': N_TRAIN,
        'n_harmful_used':    int(n_harmful),
        'n_harmless_used':   int(n_harmless),
        'mm_natural_scale':  norms.tolist(),                 # (n_layers, n_pos)
        'frac_harmful_proj_larger': frac_harmful_proj_larger.tolist(),
    },
    'eval_awareness': {
        'recipe':   'Nguyen et al. 2507.01786 — mean-diff at last A/B answer-char token',
        'datasets': {ds: {k: r[k] for k in ('n_used', 'n_skipped', 'mm_natural_scale', 'frac_test_proj_larger')}
                     for ds, r in eval_aw_results.items()},
    },
}

# Refusal arrays — keyed by (pos, layer). pos is the original negative index for clarity.
for pi, P in enumerate(POSITIONS):
    for L in range(n_layers):
        suffix = f'pos_{P:+d}__layer_{L:03d}'
        npz_payload[f'refusal__mm_dir__{suffix}']        = refusal_dir[L, pi].numpy()
        npz_payload[f'refusal__mm_raw__{suffix}']        = raw_diff[L, pi].numpy()
        npz_payload[f'refusal__harmful_mean__{suffix}']  = harmful_mean[L, pi].numpy()
        npz_payload[f'refusal__harmless_mean__{suffix}'] = harmless_mean[L, pi].numpy()

# Eval-aw arrays — keyed by (dataset, layer).
for ds_name, r in eval_aw_results.items():
    for L in range(n_layers):
        suffix = f'{ds_name}__layer_{L:03d}'
        npz_payload[f'eval_aw__mm_dir__{suffix}']      = r['direction'][L]
        npz_payload[f'eval_aw__mm_raw__{suffix}']      = (r['test_mean'][L] - r['deploy_mean'][L])
        npz_payload[f'eval_aw__test_mean__{suffix}']   = r['test_mean'][L]
        npz_payload[f'eval_aw__deploy_mean__{suffix}'] = r['deploy_mean'][L]

np.savez_compressed(OUT_DIR / 'directions.npz', **npz_payload)
(OUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2, default=float))
print(f'wrote {len(npz_payload)} arrays to {OUT_DIR / "directions.npz"}')
print(f'wrote manifest to {OUT_DIR / "manifest.json"}')

# Compactness: print byte sizes.
for f in sorted(OUT_DIR.iterdir()):
    print(f'  {f.name:<24s} {f.stat().st_size/1e6:>8.2f} MB')

## Notes & gotchas

- **Why both extractions in one notebook.** The model is the same, so loading once and running both halves saves the (small) load time + lets us write both into one NPZ that downstream comparison code can index by prefix.
- **`enable_thinking=False`.** Llama 3.3 70B's chat template emits a `<think>...</think>` block by default. With thinking on, the EOI region used by Arditi's recipe is no longer the last 5 tokens — it sits before the think block. We turn it off so the position semantics match the Arditi paper.
- **No selection step.** We saved all `(n_pos × n_layers)` candidate refusal directions rather than picking one. Downstream consumers can pick by `frac_harmful_proj_larger` (in `manifest.json`) or refit Arditi's ablation-based selection on top of the saved cube.
- **No filter step.** The original Arditi pipeline drops items that the model already refuses (harmless side) or already complies with (harmful side). We skipped this — it adds 1 extra inference pass per item and changes the direction by a small amount. To enable, run `pipeline.submodules.select_direction.get_refusal_scores` over each split and threshold; the easiest path is to re-run the full upstream pipeline.
- **Layer-alignment with `exp06_lamma/`.** The S/U directions in `exp06_lamma/directions.npz` use the same post-block hook on `model.model.layers[L]`, so `eval_aw__mm_dir__{ds}__layer_{L}` and the S/U direction at layer L are directly comparable via cosine.
- **Re-using the loaded model.** `loaded` stays in the kernel after this notebook finishes — you can chain straight into `12_refusal_direction_llama33_70b.ipynb` / `13_eval_awareness_directions_llama33_70b.ipynb` without reload.